# 08 - Transcript cleaning review

Revision de ASR2, desacuerdos, candidatos y policy para decidir E2.

In [1]:
import pandas as pd
from IPython.display import Markdown, display
from data_cleaning.src.transcript_cleaning_review import (
    cargar_outputs, resumen, cambios_por_tipo, usability_counts, disagreement_counts,
    ejemplos_autoaplicados, candidatos_por_tipo, candidatos_no_autoaplicados,
    top_fuentes_problemas, impacto_train, decision,
)
pd.set_option("display.max_colwidth", 180)
changes, candidates, policy, asr2, disagreement = cargar_outputs()

## Resumen

In [2]:
summary = resumen(changes, candidates, policy, asr2, disagreement)
display(summary)

,transcripts,auto_clean_changes,asr2_rows,asr2_ok,asr2_blocked,disagreement_rows,disagreement_high,replacement_candidates,auto_replacements,usable,questionable,bad_candidate,excluded_train
0,5950,52,5950,5950,0,5950,192,2194,0,3780,1978,192,134


## Current vs ASR2

In [3]:
if asr2.empty:
    display(Markdown("**ASR2 no disponible localmente.**"))
else:
    cols = ["source_id", "clip", "split", "current_text", "asr2_text", "asr2_model", "status", "reason", "asr2_runtime_sec"]
    display(asr2[[c for c in cols if c in asr2.columns]].head(20))

,source_id,clip,split,current_text,asr2_text,asr2_model,status,reason,asr2_runtime_sec
0,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0001,test,ok estamos en vivo acabo de subir un video nuevo a mi canal de youtube sobre el curupi,"Ok, estamos en vivo. Acabo de subir un video nuevo a mi canal de YouTube sobre el curupín.",faster-whisper:large-v3-turbo,ok,NaN,0.535
1,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0002,test,es un mito paraguayo lo deberian ver,"es un mito paraguayo, lo deberían ver.",faster-whisper:large-v3-turbo,ok,NaN,0.286
2,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0003,test,pero bueno creo que estamos con 7 dias seguidos haciendo lives,"Pero bueno, creo que estamos con 7 días seguidos haciendo live.",faster-whisper:large-v3-turbo,ok,NaN,0.292
3,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0004,test,que buena onda espero que la esten disfrutando como yo,¡Qué buena onda! Espero que la están disfrutando como yo.,faster-whisper:large-v3-turbo,ok,NaN,0.293
4,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0005,test,que haces andy todo bien como va la gente mientras todos estan entrando,¿Qué haces Andy? ¿Todo bien? ¿Cómo va la gente mientras todos están entrando?,faster-whisper:large-v3-turbo,ok,NaN,0.307
5,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0006,test,queria contarles una anecdota muy graciosa que haces moni que haces rama valen ana,Quería contarles una anécdota muy graciosa. ¿Qué haces Moni? ¿Qué haces Rama? ¿Valen? ¿Ana?,faster-whisper:large-v3-turbo,ok,NaN,0.388
6,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0007,test,que buena onda que todos entran asi tan rapido enzo maximo lorito dominic marcelo tuti,"¡Ey! ¡Qué buena onda que todos entran así tan rápido! Enzo, Máximo, Lorito, Dominic, Marcelo, Tuti.",faster-whisper:large-v3-turbo,ok,NaN,0.389
7,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0008,test,bueno vamos a arrancar con esto les queria contar una anecdota muy graciosa en mi vida,"Bueno, vamos a arrancar con esto Les quería contar una anécdota Muy graciosa en mi vida",faster-whisper:large-v3-turbo,ok,NaN,0.352
8,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0009,test,donde ser argentino o por lo menos mentir y decirles que soy argentino me salvo la vida,donde ser argentino o por lo menos mentir y decirles que soy argentino me salvó la vida,faster-whisper:large-v3-turbo,ok,NaN,0.311
9,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0010,test,basicamente una vez hace ponele no se 4 o 5 años,"Básicamente Una vez hace Ponele, no sé 4 o 5 años",faster-whisper:large-v3-turbo,ok,NaN,0.366


## WER/CER current vs ASR2

In [4]:
if disagreement.empty:
    display(Markdown("**Sin disagreement audit disponible.**"))
else:
    display(disagreement_counts(disagreement))
    cols = ["source_id", "clip", "current_text", "asr2_text", "wer_current_vs_asr2", "cer_current_vs_asr2", "token_diff_summary", "disagreement_level", "reasons"]
    display(disagreement[[c for c in cols if c in disagreement.columns]].head(20))

,disagreement_level,clips
1,low,3784
2,medium,1974
0,high,192


,source_id,clip,current_text,asr2_text,wer_current_vs_asr2,cer_current_vs_asr2,token_diff_summary,disagreement_level,reasons
0,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0001,ok estamos en vivo acabo de subir un video nuevo a mi canal de youtube sobre el curupi,"Ok, estamos en vivo. Acabo de subir un video nuevo a mi canal de YouTube sobre el curupín.",0.166667,0.072464,"replace: 'ok' -> 'ok,' | replace: 'vivo' -> 'vivo.' | replace: 'curupi' -> 'curupín.'",low,NaN
1,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0002,es un mito paraguayo lo deberian ver,"es un mito paraguayo, lo deberían ver.",0.428571,0.100000,"replace: 'paraguayo' -> 'paraguayo,' | replace: 'deberian ver' -> 'deberían ver.'",medium,possible_entity_error
2,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0003,pero bueno creo que estamos con 7 dias seguidos haciendo lives,"Pero bueno, creo que estamos con 7 días seguidos haciendo live.",0.272727,0.057692,"replace: 'bueno' -> 'bueno,' | replace: 'dias' -> 'días' | replace: 'lives' -> 'live.'",low,NaN
3,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0004,que buena onda espero que la esten disfrutando como yo,¡Qué buena onda! Espero que la están disfrutando como yo.,0.400000,0.111111,replace: 'que' -> '¡qué' | replace: 'onda' -> 'onda!' | replace: 'esten' -> 'están' | replace: 'yo' -> 'yo.',medium,possible_entity_error
4,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0005,que haces andy todo bien como va la gente mientras todos estan entrando,¿Qué haces Andy? ¿Todo bien? ¿Cómo va la gente mientras todos están entrando?,0.538462,0.152542,replace: 'que' -> '¿qué' | replace: 'andy todo bien como' -> 'andy? ¿todo bien? ¿cómo' | replace: 'estan entrando' -> 'están entrando?',medium,asr_disagreement
5,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0006,queria contarles una anecdota muy graciosa que haces moni que haces rama valen ana,Quería contarles una anécdota muy graciosa. ¿Qué haces Moni? ¿Qué haces Rama? ¿Valen? ¿Ana?,0.642857,0.188406,replace: 'queria' -> 'quería' | replace: 'anecdota' -> 'anécdota' | replace: 'graciosa que' -> 'graciosa. ¿qué' | replace: 'moni que' -> 'moni? ¿qué',medium,possible_timestamp_misalignment
6,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0007,que buena onda que todos entran asi tan rapido enzo maximo lorito dominic marcelo tuti,"¡Ey! ¡Qué buena onda que todos entran así tan rápido! Enzo, Máximo, Lorito, Dominic, Marcelo, Tuti.",0.666667,0.222222,"replace: 'que' -> '¡ey! ¡qué' | replace: 'asi' -> 'así' | replace: 'rapido enzo maximo lorito dominic marcelo tuti' -> 'rápido! enzo, máximo, lorito, dominic, marcelo, tuti.'",medium,possible_timestamp_misalignment
7,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0008,bueno vamos a arrancar con esto les queria contar una anecdota muy graciosa en mi vida,"Bueno, vamos a arrancar con esto Les quería contar una anécdota Muy graciosa en mi vida",0.187500,0.042254,"replace: 'bueno' -> 'bueno,' | replace: 'queria' -> 'quería' | replace: 'anecdota' -> 'anécdota'",low,NaN
8,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0009,donde ser argentino o por lo menos mentir y decirles que soy argentino me salvo la vida,donde ser argentino o por lo menos mentir y decirles que soy argentino me salvó la vida,0.058824,0.014085,replace: 'salvo' -> 'salvó',low,NaN
9,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0010,basicamente una vez hace ponele no se 4 o 5 años,"Básicamente Una vez hace Ponele, no sé 4 o 5 años",0.272727,0.078947,"replace: 'basicamente' -> 'básicamente' | replace: 'ponele' -> 'ponele,' | replace: 'se' -> 'sé'",low,NaN


## Cleaned text y auto-applied replacements

In [5]:
display(cambios_por_tipo(changes))
display(ejemplos_autoaplicados(changes, 15))

,change_type,clips
0,space_normalization,52


,source_id,clip,change_type,evidence,original_text,asr2_text,cleaned_text
12,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0013,space_normalization,espacios multiples/strip,tengo varios amigos que viven en montreal especialmente en montreal especialmente durante el verano en montreal,tengo varios amigos que viven en Montreal especialmente durante el verano en Montreal,tengo varios amigos que viven en montreal especialmente en montreal especialmente durante el verano en montreal
69,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0070,space_normalization,espacios multiples/strip,que lindo verte aca ah leo le respondo a esta pregunta a sebastian aguero,Qué lindo verte acá. Le respondo a esta pregunta a Sebastián Agüero.,que lindo verte aca ah leo le respondo a esta pregunta a sebastian aguero
398,ME ACUSARON DE BRUJA Y ME TUVE QUE IR DEL PUEBLO -,clip_0286,space_normalization,espacios multiples/strip,yo marcela tauro todo jugo jugo con esa carita,"yo, Marcela Tauro todos juntos, así con esa carita",yo marcela tauro todo jugo jugo con esa carita
492,ME ACUSARON DE BRUJA Y ME TUVE QUE IR DEL PUEBLO -,clip_0380,space_normalization,espacios multiples/strip,claro esta tu vida vieja y enterrada tus ancestros te la rebaja,"Claro, está tu vieja ahí enterrada, tus ancestros, te la rebaja.",claro esta tu vida vieja y enterrada tus ancestros te la rebaja
832,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0191,space_normalization,espacios multiples/strip,no se fala fala no se no se no digo amigo en que momento entramos a brasil,"y no se... Fala, José. Va a llenar. Digo, amigo, ¿en qué momento entramos a Brasil?",no se fala fala no se no se no digo amigo en que momento entramos a brasil
1014,"AZZARO REACCIÓN - RIVER, A LA FINAL LE GANÓ 1-0 A",clip_0056,space_normalization,espacios multiples/strip,es muy dificil es mucha ventaja es mucha ventaja lo del arbitro fue realmente impresentable en esa decision,es mucha ventaja es mucha ventaja lo del árbitro fue realmente impresentable en esa decisión,es muy dificil es mucha ventaja es mucha ventaja lo del arbitro fue realmente impresentable en esa decision
1113,"AZZARO REACCIÓN - RIVER, A LA FINAL LE GANÓ 1-0 A",clip_0158,space_normalization,espacios multiples/strip,a lo bueno no le sirve no le sirve a los buenos no les sirve,a lo bueno no le sirve a lo bueno no le sirve,a lo bueno no le sirve no le sirve a los buenos no les sirve
1119,"AZZARO REACCIÓN - RIVER, A LA FINAL LE GANÓ 1-0 A",clip_0164,space_normalization,espacios multiples/strip,hace cuatro meses estaba con el boleto a santiago del estero y que le importa que me importaba la mitologia,Hace cuatro meses estaba con el boleto a Santiago del Estero. ¿Y qué me importaba el ámbito?,hace cuatro meses estaba con el boleto a santiago del estero y que le importa que me importaba la mitologia
1125,"AZZARO REACCIÓN - RIVER, A LA FINAL LE GANÓ 1-0 A",clip_0170,space_normalization,espacios multiples/strip,refundemos river pero pero y hoy goza de las mieles de,refundemos River y hoy goza de las mieles de,refundemos river pero pero y hoy goza de las mieles de
1145,"AZZARO REACCIÓN - RIVER, A LA FINAL LE GANÓ 1-0 A",clip_0192,space_normalization,espacios multiples/strip,de las circunstancias venga merecidisima clasificacion de river,de las circunstancias merecidísima clasificación de River,de las circunstancias venga merecidisima clasificacion de river


## Candidates de reemplazo y desacuerdo

In [6]:
if candidates.empty:
    display(Markdown("**Sin candidates.**"))
else:
    display(candidates.groupby("candidate_type").size().rename("clips").reset_index().sort_values("clips", ascending=False))
    display(candidatos_no_autoaplicados(candidates, 30))

,candidate_type,clips
0,asr_disagreement,1877
3,possible_misalignment,235
1,possible_audio_text_mismatch,64
2,possible_hallucination,18


,source_id,clip,candidate_type,span,suggestion,evidence,confidence,reason_not_auto_applied
0,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0002,asr_disagreement,"replace: 'paraguayo' -> 'paraguayo,' | replace: 'deberian ver' -> 'deberían ver.'",NaN,level=medium; reasons=possible_entity_error; wer=0.428571; cer=0.100000,medium,requiere revision humana; no hay evidencia fuerte para reescribir
1,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0004,asr_disagreement,replace: 'que' -> '¡qué' | replace: 'onda' -> 'onda!' | replace: 'esten' -> 'están' | replace: 'yo' -> 'yo.',NaN,level=medium; reasons=possible_entity_error; wer=0.400000; cer=0.111111,medium,requiere revision humana; no hay evidencia fuerte para reescribir
2,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0005,asr_disagreement,replace: 'que' -> '¿qué' | replace: 'andy todo bien como' -> 'andy? ¿todo bien? ¿cómo' | replace: 'estan entrando' -> 'e,NaN,level=medium; reasons=asr_disagreement; wer=0.538462; cer=0.152542,medium,requiere revision humana; no hay evidencia fuerte para reescribir
3,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0006,possible_misalignment,replace: 'queria' -> 'quería' | replace: 'anecdota' -> 'anécdota' | replace: 'graciosa que' -> 'graciosa. ¿qué' | replac,NaN,level=medium; reasons=possible_timestamp_misalignment; wer=0.642857; cer=0.188406,medium,requiere revision humana; no hay evidencia fuerte para reescribir
4,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0007,possible_misalignment,replace: 'que' -> '¡ey! ¡qué' | replace: 'asi' -> 'así' | replace: 'rapido enzo maximo lorito dominic marcelo tuti' -> ',NaN,level=medium; reasons=possible_timestamp_misalignment; wer=0.666667; cer=0.222222,medium,requiere revision humana; no hay evidencia fuerte para reescribir
5,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0011,possible_misalignment,"insert: '' -> 'a... viajé' | replace: 'viaje a montreal quebec canada' -> 'montreal, quebec, canadá.'",NaN,level=medium; reasons=possible_timestamp_misalignment; wer=0.714286; cer=0.266667,medium,requiere revision humana; no hay evidencia fuerte para reescribir
6,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0012,asr_disagreement,"replace: 'frances no' -> 'francés,' | replace: 'super super' -> 'súper súper'",NaN,level=medium; reasons=possible_entity_error; wer=0.400000; cer=0.111111,medium,requiere revision humana; no hay evidencia fuerte para reescribir
7,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0014,asr_disagreement,"insert: '' -> 'montreal' | replace: 'deberian' -> 'deberían' | replace: 'montreal' -> 'montreal,' | replace: 'favor' ->",NaN,level=medium; reasons=possible_entity_error; wer=0.444444; cer=0.261905,medium,requiere revision humana; no hay evidencia fuerte para reescribir
8,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0020,asr_disagreement,"replace: 'frances' -> 'francés,' | replace: 'mas no' -> 'más' | replace: 'frances' -> 'francés'",NaN,level=medium; reasons=possible_entity_error; wer=0.444444; cer=0.125000,medium,requiere revision humana; no hay evidencia fuerte para reescribir
9,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0021,asr_disagreement,replace: 'quebecois' -> 'québécois' | replace: 'quebec' -> 'québec' | replace: 'ahi' -> 'ahí',NaN,level=medium; reasons=possible_entity_error; wer=0.375000; cer=0.105263,medium,requiere revision humana; no hay evidencia fuerte para reescribir


## Ejemplos por tipo

In [7]:
for tipo in [
    "entity_replacement_candidate",
    "slang_replacement_candidate",
    "asr_disagreement",
    "possible_hallucination",
    "possible_misalignment",
    "possible_audio_text_mismatch",
]:
    display(Markdown(f"### {tipo}"))
    display(candidatos_por_tipo(candidates, tipo, 8))

### entity_replacement_candidate

,source_id,clip,candidate_type,span,suggestion,evidence,confidence,current_text,asr2_text,reason_not_auto_applied


### slang_replacement_candidate

,source_id,clip,candidate_type,span,suggestion,evidence,confidence,current_text,asr2_text,reason_not_auto_applied


### asr_disagreement

...

,source_id,clip,candidate_type,span,suggestion,evidence,confidence,current_text,asr2_text,reason_not_auto_applied
243,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0075,possible_hallucination,asi asi asi asi,NaN,mismo token repetido 4 veces,medium,era todo tiempo todo el tiempo era asi asi asi asi,"era todo el tiempo, todo el tiempo era nasi, nasi, nasi, nasi",requiere revision humana; no hay evidencia fuerte para reescribir
260,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0118,possible_hallucination,no no no no,NaN,mismo token repetido 4 veces,medium,y ahi empezo el niño racionador ahi alan dice no no no no estos sandwiches son como,"y ahí empezó el niño racionador ahí Alan dice, no, no, no, estos son muchos, son como",requiere revision humana; no hay evidencia fuerte para reescribir
312,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0266,possible_hallucination,para para para para,NaN,mismo token repetido 4 veces,medium,no se que le dice digo para para para para digo frena el auto,"No sé qué Le dice No, no, no Digo, para, para, para, para Le digo, frena el auto",requiere revision humana; no hay evidencia fuerte para reescribir
320,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0281,possible_hallucination,segui segui segui segui,NaN,mismo token repetido 4 veces,medium,esta manejando el bueno segui segui segui segui ya esta no soy para nada,"Está manejando él Bueno, seguí, seguí, seguí, seguí Ya está, no salí para nada",requiere revision humana; no hay evidencia fuerte para reescribir
329,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0302,possible_hallucination,no no no no,NaN,mismo token repetido 4 veces,medium,imaginense marcha atras no no no no yo estaba transpirando porque se nos venian,"adelante, imagínense marcha atrás, no, no, no yo estaba transpirando porque se nos venían",requiere revision humana; no hay evidencia fuerte para reescribir
459,DAVOO XENEIZE OPINA DE BOCA 0 UNIVERSIDAD CATOLICA,clip_0008,possible_hallucination,o o o o,NaN,mismo token repetido 4 veces,medium,o o o o o o o o o o yo no encuentro palabras para explicar el fracaso que acaba de hacer boca en la copa libertadores,o creer que una derrota de Boca es más grande de lo que se quiere vender solamente para echarle leña al fuego pero opinando como hincha de Boca yo no encuentro palabras para ex...,requiere revision humana; no hay evidencia fuerte para reescribir
919,Entrevista completa por mi libro Franco con Diego,clip_0395,possible_hallucination,increible increible increible increible,NaN,mismo token repetido 4 veces,medium,gerencia general de fiat italia y conseguimos un contrato increible increible increible increible,"a la Gerencia General de Fiat Italia y conseguimos un contrato increíble, increíble, increíble.",requiere revision humana; no hay evidencia fuerte para reescribir
1193,JULI POGGIO EN FERNÉ CON GREGO,clip_0591,possible_hallucination,ya ya ya ya,NaN,mismo token repetido 4 veces,medium,liviana ya ya ya ya esta ayudando decirlo si mas alla de lo que pase despues,"más liviana. Ya está ayudando decirlo, allá de lo que pase",requiere revision humana; no hay evidencia fuerte para reescribir


### possible_misalignment

,source_id,clip,candidate_type,span,suggestion,evidence,confidence,current_text,asr2_text,reason_not_auto_applied
3,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0006,possible_misalignment,replace: 'queria' -> 'quería' | replace: 'anecdota' -> 'anécdota' | replace: 'graciosa que' -> 'graciosa. ¿qué' | replac,NaN,level=medium; reasons=possible_timestamp_misalignment; wer=0.642857; cer=0.188406,medium,queria contarles una anecdota muy graciosa que haces moni que haces rama valen ana,Quería contarles una anécdota muy graciosa. ¿Qué haces Moni? ¿Qué haces Rama? ¿Valen? ¿Ana?,requiere revision humana; no hay evidencia fuerte para reescribir
4,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0007,possible_misalignment,replace: 'que' -> '¡ey! ¡qué' | replace: 'asi' -> 'así' | replace: 'rapido enzo maximo lorito dominic marcelo tuti' -> ',NaN,level=medium; reasons=possible_timestamp_misalignment; wer=0.666667; cer=0.222222,medium,que buena onda que todos entran asi tan rapido enzo maximo lorito dominic marcelo tuti,"¡Ey! ¡Qué buena onda que todos entran así tan rápido! Enzo, Máximo, Lorito, Dominic, Marcelo, Tuti.",requiere revision humana; no hay evidencia fuerte para reescribir
5,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0011,possible_misalignment,"insert: '' -> 'a... viajé' | replace: 'viaje a montreal quebec canada' -> 'montreal, quebec, canadá.'",NaN,level=medium; reasons=possible_timestamp_misalignment; wer=0.714286; cer=0.266667,medium,fui a viaje a montreal quebec canada,"Fui a... viajé a Montreal, Quebec, Canadá.",requiere revision humana; no hay evidencia fuerte para reescribir
17,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0034,possible_misalignment,replace: 'no sabes como' -> '¿no sabés cómo' | replace: 'cambio' -> 'cambió' | replace: 'cara como que dijo eh where are,NaN,level=high; reasons=possible_timestamp_misalignment; wer=0.866667; cer=0.481481,high,no sabes como les cambio la cara como que dijo eh where are you from,¿No sabés cómo les cambió la cara? ¿Cómo dijo? ¿Eh? ¿Dónde estás?,requiere revision humana; no hay evidencia fuerte para reescribir
19,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0036,possible_misalignment,replace: 'messi oh como esta' -> '¡messi! ¡oh! ¿cómo estás?' | replace: 'cambio' -> 'cambió' | replace: 'actitud' -> 'ac,NaN,level=high; reasons=possible_timestamp_misalignment; wer=0.750000; cer=0.343750,high,messi oh como esta le cambio su actitud,¡Messi! ¡Oh! ¿Cómo estás? Le cambió su actitud.,requiere revision humana; no hay evidencia fuerte para reescribir
25,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0051,possible_misalignment,"replace: 'dice' -> 'dice,' | replace: 'chamuyo si chamuyo' -> 'chamullo. sí, chamullero.'",NaN,level=medium; reasons=possible_timestamp_misalignment;possible_entity_error; wer=0.666667; cer=0.354839,medium,natalia dice alto chamuyo si chamuyo,"Natalia dice, alto chamullo. Sí, chamullero.",requiere revision humana; no hay evidencia fuerte para reescribir
29,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0066,possible_misalignment,"replace: 'daniel daniel que haces yo' -> 'daniel, daniel, ¿qué haces? yo,' | replace: 'up man' -> 'up, man?'",NaN,level=high; reasons=possible_timestamp_misalignment; wer=0.875000; cer=0.258065,high,daniel daniel que haces yo what up man,"Daniel, Daniel, ¿qué haces? Yo, what up, man?",requiere revision humana; no hay evidencia fuerte para reescribir
30,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0067,possible_misalignment,replace: 'donde esta' -> '¿dónde está' | replace: 'kaka' -> 'kaka?',NaN,level=medium; reasons=possible_timestamp_misalignment;possible_entity_error; wer=0.600000; cer=0.190476,medium,donde esta el equipo kaka,¿Dónde está el equipo Kaka?,requiere revision humana; no hay evidencia fuerte para reescribir


### possible_audio_text_mismatch

,source_id,clip,candidate_type,span,suggestion,evidence,confidence,current_text,asr2_text,reason_not_auto_applied
20,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0037,possible_audio_text_mismatch,replace: 'y me dijo' -> 'tour' | replace: 'cien por ciento' -> '100%',NaN,level=high; reasons=possible_audio_text_mismatch; wer=0.666667; cer=0.606061,high,y me dijo por completo al cien por ciento,tour por completo al 100%,requiere revision humana; no hay evidencia fuerte para reescribir
35,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0084,possible_audio_text_mismatch,replace: 'o o o que' -> '¿qué' | replace: 'es alla' -> 'es?',NaN,level=high; reasons=possible_audio_text_mismatch; wer=0.857143; cer=0.500000,high,o o o que hora es alla,¿Qué hora es?,requiere revision humana; no hay evidencia fuerte para reescribir
67,ME ACUSARON DE BRUJA Y ME TUVE QUE IR DEL PUEBLO -,clip_0097,possible_audio_text_mismatch,yo que se en el pueblo si porque se nota y porque ven los camiones que entran y,NaN,words_per_second=6.93,medium,yo que se en el pueblo si porque se nota y porque ven los camiones que entran y se entera todo el barrio,"En el pueblo sí, porque se nota y porque ven los camiones que entran y se entran a todo el barrio.",requiere revision humana; no hay evidencia fuerte para reescribir
313,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0266,possible_audio_text_mismatch,"replace: 'se que' -> 'sé qué' | replace: 'digo' -> 'no, no, no digo, para, para, para,' | replace: 'para para para digo'",NaN,level=high; reasons=possible_audio_text_mismatch; wer=0.785714; cer=0.354167,high,no se que le dice digo para para para para digo frena el auto,"No sé qué Le dice No, no, no Digo, para, para, para, para Le digo, frena el auto",requiere revision humana; no hay evidencia fuerte para reescribir
322,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,clip_0282,possible_audio_text_mismatch,replace: 'todo esto' -> 'estoy' | replace: 'riendome' -> 'riéndome en un momento le dice bueno seguimos esos' | replace:,NaN,level=high; reasons=possible_audio_text_mismatch; wer=0.722222; cer=0.579710,high,que esto que el otro yo todo esto medio riendome 20 kilometros y el chabon mira el gps,que esto que el otro yo estoy medio riéndome en un momento le dice bueno seguimos esos 20 kilómetros y el chabón mira el GPS,requiere revision humana; no hay evidencia fuerte para reescribir
349,"AZZARO REACCIÓN - RIVER, A LA FINAL LE GANÓ 1-0 A",clip_0025,possible_audio_text_mismatch,"insert: '' -> 'difícil,' | replace: 'salto hoy almiron retrocedio' -> 'salto. oigan, miró un retrocedido.'",NaN,level=high; reasons=possible_audio_text_mismatch; wer=0.750000; cer=0.459459,high,no pudo dar ese salto hoy almiron retrocedio,"Difícil, no pudo dar ese salto. Oigan, miró un retrocedido.",requiere revision humana; no hay evidencia fuerte para reescribir
424,Coronavirus conferencia completa de Alberto Fernán,clip_0003,possible_audio_text_mismatch,replace: 'y con los expertos en infectologia y los expertos medicos' -> 'gracias.',NaN,level=high; reasons=possible_audio_text_mismatch; wer=1.000000; cer=0.895833,high,y con los expertos en infectologia y los expertos medicos,Gracias.,requiere revision humana; no hay evidencia fuerte para reescribir
429,Coronavirus conferencia completa de Alberto Fernán,clip_0037,possible_audio_text_mismatch,insert: '' -> 'dar' | replace: 'necesitan lo que esta claro es que la educacion primaria' -> 'necesitan.',NaN,level=high; reasons=possible_audio_text_mismatch; wer=0.733333; cer=0.650794,high,a sus alumnos que lo necesitan lo que esta claro es que la educacion primaria,dar a sus alumnos que lo necesitan.,requiere revision humana; no hay evidencia fuerte para reescribir


## Questionable / bad candidates

In [8]:
display(usability_counts(policy))
display(policy[policy["transcript_usability"].isin(["questionable", "bad_candidate"])].head(30))

,transcript_usability,clips
2,usable,3780
1,questionable,1978
0,bad_candidate,192


,source_id,clip,split,transcript_usability,transcript_reasons,transcript_policy_moderate
1,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0002,test,questionable,possible_entity_error,keep
3,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0004,test,questionable,possible_entity_error,keep
4,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0005,test,questionable,asr_disagreement,keep
5,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0006,test,questionable,possible_timestamp_misalignment,keep
6,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0007,test,questionable,possible_timestamp_misalignment,keep
10,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0011,test,questionable,possible_timestamp_misalignment,keep
11,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0012,test,questionable,possible_entity_error,keep
13,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0014,test,questionable,possible_entity_error,keep
19,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0020,test,questionable,possible_entity_error,keep
20,LE DIJE QUE SOY ARGENTINO - Story Time - CAP 91,clip_0021,test,questionable,possible_entity_error,keep


## Top source_ids con mas problemas

In [9]:
display(top_fuentes_problemas(policy, 15))

,source_id,problem_clips
5,DAVOO XENEIZE OPINA DE BOCA 0 UNIVERSIDAD CATOLICA,391
8,JULI POGGIO EN FERNÉ CON GREGO,250
14,ME ACUSARON DE BRUJA Y ME TUVE QUE IR DEL PUEBLO -,167
3,CHARLA SOBRE EL AMOR Y EL DESAMOR,146
11,Las y los estudiantes al frente - Entrevista a la,143
0,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,128
7,Entrevista completa por mi libro Franco con Diego,112
15,Manipulación mental Cómo tu CEREBRO literalmente c,99
13,Los lujos de L-Gante y qué aprendió cuando estuvo,80
2,"AZZARO REACCIÓN - RIVER, A LA FINAL LE GANÓ 1-0 A",79


## Impacto en train

In [10]:
display(impacto_train(policy))

,train_transcripts,excluded_bad_candidate,kept,excluded_pct
0,4826,134,4692,2.78


## Decision

In [11]:
display(Markdown("**" + decision(changes, candidates, policy, asr2, disagreement) + "**"))

**READY_FOR_VM**